In [ ]:
import os
import shutil
import random
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_DIR = "data/MSD_pancreas"
IMG_DIR = os.path.join(BASE_DIR, "3D_scans")
LBL_DIR = os.path.join(BASE_DIR, "labels")
OUTPUT_ROOT = "data/MSD_pancreas"

# Seuils pour ignorer les artefacts de segmentation (poussières et bords extrêmes)
MIN_TUMOR_PIXELS = 400
MIN_PANCREAS_PIXELS = 1000

def apply_windowing(img_array, center, width):
    """Fenêtrage radiologique dynamique -> uint8."""
    img_min = center - (width / 2.0)
    img_max = center + (width / 2.0)
    img_array = np.clip(img_array, img_min, img_max)
    img_array = (img_array - img_min) / (img_max - img_min) * 255.0
    return img_array.astype(np.uint8)

# ==========================================
# 2. FONCTION D'EXTRACTION OPTIMISÉE
# ==========================================
def extract_pseudo_3d(patient_list, target_total, folder_name, tumor_ratio=0.50, max_slices_per_patient=15):
    """Extraction avec Double Fenêtrage (Global + Focal) et filtrage du bruit."""
    
    out_img_dir = os.path.join(OUTPUT_ROOT, folder_name, "images")
    out_mask_dir = os.path.join(OUTPUT_ROOT, folder_name, "masks")
    if os.path.exists(out_img_dir): shutil.rmtree(out_img_dir)
    if os.path.exists(out_mask_dir): shutil.rmtree(out_mask_dir)
    os.makedirs(out_img_dir)
    os.makedirs(out_mask_dir)

    tumor_candidates = defaultdict(list)
    healthy_candidates = defaultdict(list)

    # 1. Catalogage groupé par patient avec filtrage des artefacts
    for p_name in tqdm(patient_list, desc=f"Cataloging {folder_name}"):
        lbl_path = os.path.join(LBL_DIR, p_name)
        if not os.path.exists(lbl_path): continue
        lbl = nib.load(lbl_path).get_fdata()
        
        for z in range(lbl.shape[2]):
            slice_mask = lbl[:, :, z]
            t_pixels = np.sum(slice_mask == 2)
            p_pixels = np.sum(slice_mask == 1)
            
            # On ne garde que les coupes avec un signal visuel exploitable
            if t_pixels >= MIN_TUMOR_PIXELS:
                tumor_candidates[p_name].append((p_name, z))
            elif p_pixels >= MIN_PANCREAS_PIXELS and t_pixels == 0:
                healthy_candidates[p_name].append((p_name, z))

    # 2. Quotas Globaux
    target_tumor = int(target_total * tumor_ratio)
    actual_healthy = target_total - target_tumor
    
    random.seed(42)
    
    # 3. Sampling intelligent avec limite par patient
    def sample_from_dict(candidate_dict, target_count):
        selected = []
        patients = list(candidate_dict.keys())
        while len(selected) < target_count and any(len(v) > 0 for v in candidate_dict.values()):
            random.shuffle(patients)
            for p in patients:
                if len(selected) >= target_count: break
                if len(candidate_dict[p]) > 0:
                    slice_info = random.choice(candidate_dict[p])
                    candidate_dict[p].remove(slice_info)
                    
                    current_p_count = sum(1 for s in selected if s[0] == p)
                    if current_p_count < max_slices_per_patient:
                        selected.append(slice_info)
        return selected

    selected_tumors = sample_from_dict(tumor_candidates, target_tumor)
    selected_healthy = sample_from_dict(healthy_candidates, actual_healthy)
    
    actual_tumor_count = len(selected_tumors)
    all_selected = selected_tumors + selected_healthy
    
    slices_by_patient = defaultdict(list)
    for p_name, s_idx in all_selected:
        slices_by_patient[p_name].append(s_idx)

    # 4. Extraction finale avec Double Fenêtrage
    saved_count = 0
    for p_name, slices in tqdm(slices_by_patient.items(), desc=f"Extracting {folder_name}"):
        img = nib.load(os.path.join(IMG_DIR, p_name)).get_fdata()
        lbl = nib.load(os.path.join(LBL_DIR, p_name)).get_fdata()
        max_z = img.shape[2] - 1
        
        for idx in slices:
            # Gestion du bord Z+1
            idx_next = idx + 1 if idx < max_z else idx - 1
            
            # --- APPLICATION DU DOUBLE FENÊTRAGE ---
            # Canal R (0) : Contexte anatomique global
            ch_global_z = apply_windowing(img[:, :, idx], center=40, width=400)
            
            # Canal G (1) : Contraste focal sur la tumeur (coupe Z)
            ch_focal_z = apply_windowing(img[:, :, idx], center=50, width=150)
            
            # Canal B (2) : Contraste focal sur la tumeur (coupe Z+1 pour la persistance 3D)
            ch_focal_z_next = apply_windowing(img[:, :, idx_next], center=50, width=150)
            
            # Stack RGB: [Global Z, Focal Z, Focal Z+1]
            slice_rgb = np.stack([ch_global_z, ch_focal_z, ch_focal_z_next], axis=-1)
            
            # Conversion pour OpenCV (écrit en BGR, relu en RGB par PIL)
            slice_bgr = cv2.cvtColor(slice_rgb, cv2.COLOR_RGB2BGR)
            
            mask_2d = lbl[:, :, idx].astype(np.uint8)
            
            fname = f"{p_name.replace('.nii.gz', '')}_s{idx}.png"
            cv2.imwrite(os.path.join(out_img_dir, fname), slice_bgr)
            cv2.imwrite(os.path.join(out_mask_dir, fname), mask_2d)
            saved_count += 1
            
    print(f"Bilan {folder_name.upper()} : {saved_count} images | Tumeurs : {actual_tumor_count}")

# ==========================================
# 3. EXÉCUTION
# ==========================================
patients = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".nii.gz")])
train_p, temp_p = train_test_split(patients, test_size=0.2, random_state=42)
val_p, test_p = train_test_split(temp_p, test_size=0.5, random_state=42)

extract_pseudo_3d(train_p, 800, "train", max_slices_per_patient=15)
extract_pseudo_3d(val_p, 100, "val", max_slices_per_patient=10)
extract_pseudo_3d(test_p, 100, "test", max_slices_per_patient=10)

In [ ]:
from pathlib import Path

# Chemin vers la racine de ton nouveau dataset
root_path = Path("data/MSD_pancreas")

for split in ["train", "val", "test"]:
    # On cible le dossier 'images' dans chaque sous-dossier
    img_folder = root_path / split / "images"
    
    if img_folder.exists():
        # On liste les fichiers .png
        count = len(list(img_folder.glob("*.png")))
        print(f"Set {split.upper():<5} : {count} images")
    else:
        print(f"Set {split.upper():<5} : Dossier introuvable")

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

def create_master_json(root_dir):
    root = Path(root_dir)
    
    # On définit désormais les deux catégories cibles
    master_data = {
        "categories": [
            {"id": 1, "name": "pancreas"},
            {"id": 2, "name": "tumor"}
        ],
        "images": [],
        "annotations": []
    }

    ann_id = 0
    img_id = 0

    for split in ["train", "val", "test"]:
        img_dir = root / split / "images"
        mask_dir = root / split / "masks"
        
        if not img_dir.exists():
            print(f"Passage du split {split} (dossier introuvable)")
            continue
            
        images = sorted(list(img_dir.glob("*.png")))
        print(f"Traitement du split {split} ({len(images)} images)...")

        for img_path in tqdm(images):
            img_cv = cv2.imread(str(img_path))
            if img_cv is None: continue
            height, width = img_cv.shape[:2]
            
            # 1. Enregistrement de l'image
            master_data["images"].append({
                "id": img_id,
                "file_name": f"{split}/images/{img_path.name}",
                "width": width,
                "height": height,
                "split": split
            })

            mask_path = mask_dir / img_path.name
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None: 
                img_id += 1
                continue

            # 2. Extraction des annotations pour chaque catégorie
            # ID 1: Pancreas, ID 2: Tumor
            for cat_id in [1, 2]:
                coords = np.column_stack(np.where(mask == cat_id))
                
                if coords.size > 0:
                    y_min, x_min = coords.min(axis=0)
                    y_max, x_max = coords.max(axis=0)
                    
                    w = int(x_max - x_min)
                    h = int(y_max - y_min)
                    
                    # On ajoute une petite marge de 1 pixel pour éviter les boîtes de surface nulle
                    w = max(1, w)
                    h = max(1, h)
                    
                    master_data["annotations"].append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": cat_id,
                        "bbox": [int(x_min), int(y_min), w, h],
                        "area": w * h,
                        "iscrowd": 0
                    })
                    ann_id += 1
            
            img_id += 1

    output_path = root / "annotations.json"
    with open(output_path, "w") as f:
        json.dump(master_data, f, indent=4)
    print(f"\nFichier créé : {output_path}")
    print(f"Total : {img_id} images et {ann_id} annotations enregistrées.")

if __name__ == "__main__":
    create_master_json("data/MSD_pancreas")

In [ ]:
import json
from collections import Counter

# Chemin vers votre fichier
json_path = "data/MSD_pancreas/annotations.json"

with open(json_path, 'r') as f:
    data = json.load(f)

# Mapping pour les noms de catégories
cat_map = {cat['id']: cat['name'] for cat in data['categories']}

# Dictionnaires pour compter
split_counts = Counter()
annotation_counts = {split: Counter() for split in ["train", "val", "test"]}

# 1. Compter les images par split
for img in data['images']:
    split_counts[img['split']] += 1

# 2. Compter les annotations par catégorie et par split
# Création d'un index pour l'id de l'image -> split
img_id_to_split = {img['id']: img['split'] for img in data['images']}

for ann in data['annotations']:
    split = img_id_to_split[ann['image_id']]
    cat_name = cat_map[ann['category_id']]
    annotation_counts[split][cat_name] += 1

# 3. Affichage des résultats
print(f"{'Split':<10} | {'Images':<8} | {'Pancréas':<10} | {'Tumeur':<8} | {'Total Ann.'}")
print("-" * 55)
for split in ["train", "val", "test"]:
    n_img = split_counts[split]
    n_panc = annotation_counts[split]['pancreas']
    n_tum = annotation_counts[split]['tumor']
    total_ann = n_panc + n_tum
    print(f"{split:<10} | {n_img:<8} | {n_panc:<10} | {n_tum:<8} | {total_ann}")

In [ ]:
import json
from collections import defaultdict

def find_images_without_pancreas(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # 1. On regroupe les catégories détectées par image_id
    # image_to_cats[id] contiendra un set des category_id présents (ex: {1, 2} ou {1})
    image_to_cats = defaultdict(set)
    for ann in data['annotations']:
        image_to_cats[ann['image_id']].add(ann['category_id'])

    # 2. On crée un index pour retrouver les informations de l'image par son ID
    id_to_info = {img['id']: img for img in data['images']}

    print("Recherche des coupes avec tumeur (ID 2) mais sans pancréas sain (ID 1)...")
    print("-" * 50)
    
    count = 0
    for img_id, cats in image_to_cats.items():
        # Condition : La tumeur est présente (2) mais le pancréas sain est absent (1)
        if 2 in cats and 1 not in cats:
            info = id_to_info[img_id]
            print(f"ID Image : {img_id}")
            print(f"Fichier  : {info['file_name']}")
            print(f"Split    : {info['split']}")
            print("-" * 30)
            count += 1

    print(f"Recherche terminée. {count} image(s) trouvée(s).")

if __name__ == "__main__":
    # Assure-toi que le chemin correspond bien à ton arborescence
    find_images_without_pancreas("data/MSD_pancreas/annotations.json")

In [ ]:
import json
from pathlib import Path

def analyze_tumor_distribution(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # 1. Créer un set des IDs d'images contenant une tumeur (catégorie 2)
    images_with_tumor_ids = {
        ann['image_id'] for ann in data['annotations'] 
        if ann['category_id'] == 2
    }

    # 2. Initialiser les compteurs
    # Structure : { 'train': {'total': 0, 'tumor': 0}, ... }
    stats = {
        split: {'total': 0, 'with_tumor': 0} 
        for split in ['train', 'val', 'test']
    }

    # 3. Parcourir les images et compter
    for img in data['images']:
        split = img['split']
        stats[split]['total'] += 1
        if img['id'] in images_with_tumor_ids:
            stats[split]['with_tumor'] += 1

    # 4. Affichage des résultats
    print(f"{'Split':<10} | {'Total':<10} | {'Avec Tumeur':<15} | {'Sans Tumeur':<15} | % Tumeur")
    print("-" * 70)
    
    for split, counts in stats.items():
        total = counts['total']
        with_t = counts['with_tumor']
        no_t = total - with_t
        perc = (with_t / total * 100) if total > 0 else 0
        
        print(f"{split.upper():<10} | {total:<10} | {with_t:<15} | {no_t:<15} | {perc:.1f}%")

if __name__ == "__main__":
    json_file = "data/MSD_pancreas/annotations.json"
    if Path(json_file).exists():
        analyze_tumor_distribution(json_file)
    else:
        print(f"Erreur : Le fichier {json_file} est introuvable.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img_path = "data/MSD_pancreas/test/images/pancreas_045_s21.png"
mask_path = "data/MSD_pancreas/test/masks/pancreas_045_s21.png"

# 1. Charger l'image et convertir en RGB
image_bgr = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# 2. Charger le masque en niveaux de gris (1 pixel = 1 ID de classe)
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

# 3. Création de la figure
plt.figure(figsize=(12, 6))

# Sous-intrigue 1 : Image brute
plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Scanner (CT) original")
plt.axis("off")

# Sous-intrigue 2 : Superposition
plt.subplot(1, 2, 2)
plt.imshow(image_rgb)

# On affiche le masque avec une transparence de 40% (alpha=0.4)
# 'jet' permet de colorer le Pancréas (1) et la Tumeur (2) différemment
# On utilise masked_where pour ne pas colorer le fond (0)
masked_data = np.ma.masked_where(mask == 0, mask)
plt.imshow(masked_data, alpha=0.5, cmap="jet", interpolation='none')

plt.title("Overlay : Pancréas et Tumeur")
plt.axis("off")

plt.tight_layout()
plt.show()